In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

In [2]:
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
BASE_PATH = "/content/"

X_train_path = os.path.join(BASE_PATH, "X_train.txt")
y_train_path = os.path.join(BASE_PATH, "y_train.txt")

X_test_path = os.path.join(BASE_PATH, "X_test.txt")
y_test_path = os.path.join(BASE_PATH, "y_test.txt")

activity_labels_path = os.path.join(BASE_PATH, "activity_labels.txt")

In [ ]:
X_train = pd.read_csv(X_train_path, sep=r"\s+", header=None).values
y_train = pd.read_csv(y_train_path, sep=r"\s+", header=None).values.ravel()
X_test = pd.read_csv(X_test_path, sep=r"\s+", header=None).values
y_test = pd.read_csv(y_test_path, sep=r"\s+", header=None).values.ravel()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (7352, 561)
y_train: (7352,)
X_test: (2947, 561)
y_test: (2947,)


In [ ]:
activity_labels = pd.read_csv(activity_labels_path, sep=r"\s+", header=None, names=["id", "activity"])
activity_labels

,id,activity
0,1,WALKING
1,2,WALKING_UPSTAIRS
2,3,WALKING_DOWNSTAIRS
3,4,SITTING
4,5,STANDING
5,6,LAYING


In [ ]:
activity_map = dict(zip(activity_labels["id"], activity_labels["activity"]))
print(activity_map)

{1: 'WALKING', 2: 'WALKING_UPSTAIRS', 3: 'WALKING_DOWNSTAIRS', 4: 'SITTING', 5: 'STANDING', 6: 'LAYING'}


In [7]:
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Encoded classes:")
print(label_encoder.classes_)

Encoded classes:
[1 2 3 4 5 6]


In [8]:
with open("label_encoder.pkl", "wb") as file:
    pickle.dump(label_encoder, file)

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

In [ ]:
X_train_rnn = X_train_scaled.reshape(
    X_train_scaled.shape[0], X_train_scaled.shape[1], 1
)

X_test_rnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

print("X_train RNN shape:", X_train_rnn.shape)
print("X_test RNN shape:", X_test_rnn.shape)

X_train RNN shape: (7352, 561, 1)
X_test RNN shape: (2947, 561, 1)


In [ ]:
model = Sequential(
    [
        SimpleRNN(128, activation="tanh", input_shape=(561, 1), return_sequences=True),
        Dropout(0.3),
        SimpleRNN(64, activation="tanh", return_sequences=False),
        Dropout(0.3),
        Dense(32, activation="relu"),
        Dropout(0.2),
        Dense(6, activation="softmax"),
    ]
)

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 561, 128)       │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 561, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,270 (122.15 KB)

 Trainable params: 31,270 (122.15 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)

In [ ]:
log_dir = "logs/fit"
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stopping = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train_rnn,
    y_train_encoded,
    validation_split=0.20,
    epochs=20,
    batch_size=64,
    callbacks=[early_stopping, tensorboard_callback],
    verbose=1,
)

Epoch 1/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 54s 558ms/step - accuracy: 0.5848 - loss: 0.9999 - val_accuracy: 0.7070 - val_loss: 0.6582
Epoch 2/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 48s 523ms/step - accuracy: 0.6038 - loss: 0.9433 - val_accuracy: 0.6084 - val_loss: 0.9648
Epoch 3/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 83s 540ms/step - accuracy: 0.5861 - loss: 1.0210 - val_accuracy: 0.6125 - val_loss: 0.9322
Epoch 4/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 82s 533ms/step - accuracy: 0.6511 - loss: 0.8178 - val_accuracy: 0.7328 - val_loss: 0.6443
Epoch 5/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 49s 539ms/step - accuracy: 0.7393 - loss: 0.6061 - val_accuracy: 0.7301 - val_loss: 0.5819
Epoch 6/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 50s 543ms/step - accuracy: 0.7592 - loss: 0.5817 - val_accuracy: 0.7179 - val_loss: 0.6658
Epoch 7/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 82s 548ms/step - accuracy: 0.7041 - loss: 0.7222 - val_accuracy: 0.7172 - val_loss: 0.6181
Epoch 8/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 48s 518ms/step - accuracy: 0.7706 - loss: 0.5478 - val_accu

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test_rnn, y_test_encoded, verbose=1)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

93/93 ━━━━━━━━━━━━━━━━━━━━ 7s 79ms/step - accuracy: 0.7465 - loss: 0.6247
Test Loss: 0.6247040629386902
Test Accuracy: 0.7465218901634216


In [ ]:
y_probability = model.predict(X_test_rnn, verbose=1)
y_pred = np.argmax(y_probability, axis=1)
print("Prediction shape:", y_pred.shape)

93/93 ━━━━━━━━━━━━━━━━━━━━ 8s 78ms/step
Prediction shape: (2947,)


In [ ]:
model.save("model.keras")
print("Model saved successfully.")

Model saved successfully.


In [ ]:
sample_index = 0
sample = X_test_rnn[sample_index : sample_index + 1]
prediction = model.predict(sample, verbose=0)
predicted_class = np.argmax(prediction, axis=1)[0]
predicted_activity = activity_names[predicted_class]
actual_class = y_test_encoded[sample_index]
actual_activity = activity_names[actual_class]
confidence = np.max(prediction)
print("Actual Activity:", actual_activity)
print("Predicted Activity:", predicted_activity)
print(f"Confidence: {confidence * 100:.2f}%")

Actual Activity: STANDING
Predicted Activity: STANDING
Confidence: 96.68%
